# Lab 2.1 (ต่อ) · Profiling และทำความสะอาดข้อมูลลูกค้า `customers.xlsx`

ข้อมูล `customers.xlsx` คือรายชื่อสมาชิกของร้านบ้านบรู (1 แถว = ลูกค้า 1 คน) ใช้คู่กับ `customer_id` ในข้อมูลยอดขาย
เป้าหมายคือ **รายงานปัญหาให้ครบก่อน (ขั้น 1–5) ตัดสินใจ (ขั้น 6) แล้วค่อยแก้ (ขั้น 7–9)** ได้ `customers_clean.csv` พร้อม log

| ขั้น | ทำอะไร |
|---|---|
| 0 | เตรียมเครื่องมือ |
| 1 | โหลดและดูข้อมูลครั้งแรก |
| 2 | ภาพรวมทุกคอลัมน์: ค่าว่าง ค่าไม่ซ้ำ รูปแบบ |
| 3 | ตรวจทีละคอลัมน์ |
| 4 | ตรวจข้ามไฟล์กับ `branches.xlsx` และข้อมูลยอดขาย |
| 5 | สรุปปัญหาเป็นตาราง |
| 6 | ตัดสินใจก่อนลงมือ |
| 7 | ทำความสะอาดตามที่ตัดสินใจ |
| 8 | Verify |
| 9 | Export `customers_clean.csv` และ log |

**ไฟล์ที่ใช้:** `customers.xlsx` (ต้องมี) · `branches.xlsx` และ `sales_clean.csv` (ถ้ามี จะได้ตรวจข้ามไฟล์ในขั้นที่ 4)

## ขั้นที่ 0 · เตรียมเครื่องมือ
รันเซลล์นี้ก่อน (ไม่ต้องแก้) มีฟังก์ชัน `load_file()` ที่ให้อัปโหลดไฟล์เมื่อยังไม่มีใน Colab และ `pattern()` ที่ใช้ดูรูปแบบของค่า

In [1]:
# ===== เซลล์ตัวช่วย · รันก่อน ไม่ต้องแก้ =====
import os, re
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)


def load_file(name, required=True):
    """อ่านไฟล์ (.xlsx / .csv) ทุกคอลัมน์เป็นข้อความ ค่าว่างเป็น "" ถ้าไม่มีไฟล์จะให้อัปโหลด (Google Colab)"""
    if not os.path.exists(name):
        try:
            from google.colab import files
            print(f"ไม่พบ {name} กรุณาอัปโหลดไฟล์" + ("" if required else " (กด Cancel เพื่อข้ามได้)"))
            files.upload()
        except ImportError:
            pass
    if not os.path.exists(name):
        if required:
            raise FileNotFoundError(f"ไม่พบไฟล์ {name} ให้วางไฟล์ไว้โฟลเดอร์เดียวกับ notebook")
        print(f"ข้าม {name}: ไม่มีไฟล์")
        return None
    if name.endswith(".xlsx"):
        return pd.read_excel(name, dtype=str, keep_default_na=False)
    return pd.read_csv(name, dtype=str, keep_default_na=False)


def pattern(value):
    """แทนตัวเลขด้วย 9 และตัวอักษรไทยด้วย ก เพื่อดูว่าค่าในคอลัมน์มีกี่รูปแบบ เช่น C00012 -> C99999"""
    return re.sub(r"[ก-๙]+", "ก", re.sub(r"\d", "9", value))


issues = []  # (คอลัมน์, ปัญหา, จำนวนแถว, ความรุนแรง, หมายเหตุ) ใช้สรุปในขั้นที่ 5
def note(column, problem, n, severity, detail=""):
    issues.append({"คอลัมน์": column, "ปัญหา": problem, "จำนวนแถว": int(n), "ความรุนแรง": severity, "หมายเหตุ": detail})

## ขั้นที่ 1 · ดูข้อมูลครั้งแรก

อ่าน **ทุกคอลัมน์เป็นข้อความ** โดยตั้งใจ เพื่อไม่ให้ pandas แปลงชนิดให้เองจนซ่อนปัญหา
ข้อควรรู้: Excel เก็บ `joined_date` เป็นชนิดวันที่ เมื่ออ่านเป็นข้อความจึงได้ `YYYY-MM-DD 00:00:00`

In [2]:
df = load_file("customers.xlsx")
print(f"{len(df):,} แถว × {df.shape[1]} คอลัมน์")
df.head(8)

3,000 แถว × 7 คอลัมน์


,customer_id,nickname,gender,age_group,home_branch_id,joined_date,phone
0,C00001,กมล,หญิง,18-24,B02,2026-03-03 00:00:00,082-xxx-6693
1,C00002,กิตติ,ชาย,25-34,B04,2026-01-18 00:00:00,084-xxx-3734
2,C00003,ปิยะ,หญิง,18-24,B04,2025-05-04 00:00:00,089-xxx-9286
3,C00004,ปิยะ,หญิง,18-24,B05,2026-05-05 00:00:00,083-xxx-7314
4,C00005,รัตนา,หญิง,25-34,B04,2026-07-10 00:00:00,083-xxx-5701
5,C00006,กมล,ชาย,18-24,B04,2026-08-16 00:00:00,086-xxx-1895
6,C00007,วรรณ,ชาย,45-54,B03,2026-07-17 00:00:00,087-xxx-3147
7,C00008,ภัทร,หญิง,25-34,B01,2026-04-07 00:00:00,087-xxx-9877


**คำถามชวนคิด:** ข้อมูลยอดขายมีลูกค้าสมาชิกไม่ซ้ำ 2,507 คน แต่ไฟล์นี้มีกี่คน และส่วนต่างหมายถึงอะไร

## ขั้นที่ 2 · ภาพรวมทุกคอลัมน์

`รูปแบบ` บอกว่าค่าในคอลัมน์มีหน้าตากี่แบบ ถ้าคอลัมน์ไหนมีหลายรูปแบบ แปลว่ามีค่าที่หน้าตาผิดจากคนอื่นปนอยู่

In [3]:
overview = pd.DataFrame({
    "ค่าว่าง": (df.apply(lambda s: s.str.strip()) == "").sum(),
    "ค่าไม่ซ้ำ": df.nunique(),
    "จำนวนรูปแบบ": df.apply(lambda s: s.map(pattern).nunique()),
    "รูปแบบที่พบบ่อย": df.apply(lambda s: ", ".join(s.map(pattern).value_counts().head(3).index)),
    "ตัวอย่าง": df.apply(lambda s: ", ".join(s.drop_duplicates().head(3))),
})
n_dup_rows = int(df.duplicated().sum())
n_space = int((df != df.apply(lambda s: s.str.strip())).any(axis=1).sum())
print(f"แถวที่ซ้ำกันทุกคอลัมน์: {n_dup_rows:,} · แถวที่มีช่องว่างหัว/ท้าย: {n_space:,}")
note("(ทุกคอลัมน์)", "ค่าว่าง", overview["ค่าว่าง"].sum(), "—", "ไม่มีค่าว่างเลย" if overview["ค่าว่าง"].sum() == 0 else "")
note("(ทุกคอลัมน์)", "แถวซ้ำทุกคอลัมน์", n_dup_rows, "—" if n_dup_rows == 0 else "สูง")
overview

แถวที่ซ้ำกันทุกคอลัมน์: 0 · แถวที่มีช่องว่างหัว/ท้าย: 0


,ค่าว่าง,ค่าไม่ซ้ำ,จำนวนรูปแบบ,รูปแบบที่พบบ่อย,ตัวอย่าง
customer_id,0,3000,1,C99999,"C00001, C00002, C00003"
nickname,0,24,1,ก,"กมล, กิตติ, ปิยะ"
gender,0,3,1,ก,"หญิง, ชาย, ไม่ระบุ"
age_group,0,6,3,"99-99, 99+, ก 99","18-24, 25-34, 45-54"
home_branch_id,0,5,1,B99,"B02, B04, B05"
joined_date,0,530,1,9999-99-99 99:99:99,"2026-03-03 00:00:00, 2026-01-18 00:00:00, 2025..."
phone,0,2950,1,999-xxx-9999,"082-xxx-6693, 084-xxx-3734, 089-xxx-9286"


## ขั้นที่ 3 · ตรวจทีละคอลัมน์

### 3.1 `customer_id` · ต้องไม่ซ้ำ และรูปแบบ `C` + ตัวเลข 5 หลัก

In [4]:
bad_id = df[~df["customer_id"].str.fullmatch(r"C\d{5}")]
n_dup_id = int(df["customer_id"].duplicated().sum())
print(f"รูปแบบผิด: {len(bad_id):,} · id ซ้ำ: {n_dup_id:,} · ช่วง {df['customer_id'].min()} – {df['customer_id'].max()}")
note("customer_id", "รูปแบบผิดจาก C99999", len(bad_id), "—" if bad_id.empty else "สูง")
note("customer_id", "id ซ้ำ", n_dup_id, "—" if n_dup_id == 0 else "สูง")

รูปแบบผิด: 0 · id ซ้ำ: 0 · ช่วง C00001 – C03000


### 3.2 คอลัมน์แบบหมวดหมู่ · `gender`, `age_group`, `home_branch_id`
ดูค่าทั้งหมดพร้อมจำนวน (`repr` ทำให้เห็นช่องว่างที่ซ่อนอยู่)

In [5]:
for col in ["gender", "age_group", "home_branch_id"]:
    print(f"\n{col}")
    print(df[col].map(repr).value_counts().to_string())


gender
gender
'หญิง'       1652
'ชาย'        1232
'ไม่ระบุ'     116

age_group
age_group
'25-34'         993
'18-24'         775
'35-44'         577
'45-54'         346
'55+'           196
'ต่ำกว่า 18'    113

home_branch_id
home_branch_id
'B01'    776
'B05'    693
'B02'    643
'B04'    483
'B03'    405


In [6]:
n_minor = int((df["age_group"] == "ต่ำกว่า 18").sum())
n_gender_unknown = int((df["gender"] == "ไม่ระบุ").sum())
note("age_group", "ลูกค้าอายุต่ำกว่า 18 ปี", n_minor, "ต่ำ",
     "ไม่ใช่ข้อมูลผิด แต่เป็นผู้เยาว์ ต้องดูเรื่องความยินยอมตาม PDPA")
note("age_group", "ค่าเป็นข้อความ เรียงตามตัวอักษรไม่ถูก", df["age_group"].nunique(), "ต่ำ",
     "'ต่ำกว่า 18' จะไปอยู่ท้ายสุด ต้องกำหนดลำดับเองตอนทำกราฟ")
note("gender", "ไม่ระบุเพศ", n_gender_unknown, "ต่ำ", "ค่าที่ตั้งใจให้เลือกได้ ไม่ใช่ข้อมูลหาย")

### 3.3 `joined_date` · แปลงเป็นวันที่ได้ไหม อยู่ในช่วงที่สมเหตุสมผลไหม
ร้านเปิดสาขาแรก 1 มิ.ย. 2023 และข้อมูลยอดขายถึง 20 ก.ย. 2026 วันสมัครจึงไม่ควรอยู่นอกช่วงนี้ และต้องไม่เป็นปี พ.ศ.

In [7]:
jd = pd.to_datetime(df["joined_date"], errors="coerce")
n_bad_date = int(jd.isna().sum())
n_be = int((jd.dt.year > 2400).sum())
n_out = int(((jd < "2023-06-01") | (jd > "2026-09-20")).sum())
n_time = int((jd.dt.normalize() != jd).sum())
print(f"แปลงไม่ได้: {n_bad_date:,} · ปี พ.ศ.: {n_be:,} · นอกช่วง 2023-06-01 ถึง 2026-09-20: {n_out:,} · มีเวลาไม่ใช่ 00:00: {n_time:,}")
print(f"ช่วงวันสมัคร: {jd.min():%Y-%m-%d} ถึง {jd.max():%Y-%m-%d}")
print("\nจำนวนสมัครรายเดือน")
print(jd.dt.to_period("M").value_counts().sort_index().to_string())
note("joined_date", "แปลงเป็นวันที่ไม่ได้ / ปี พ.ศ. / นอกช่วง", n_bad_date + n_be + n_out, "—" if n_bad_date + n_be + n_out == 0 else "สูง")
note("joined_date", "เก็บเป็น datetime มีเวลา 00:00:00 ติดมา", len(df), "ต่ำ", "เวลาไม่มีความหมาย ควรเก็บเป็นวันที่อย่างเดียว")

แปลงไม่ได้: 0 · ปี พ.ศ.: 0 · นอกช่วง 2023-06-01 ถึง 2026-09-20: 0 · มีเวลาไม่ใช่ 00:00: 0
ช่วงวันสมัคร: 2025-04-01 ถึง 2026-09-14

จำนวนสมัครรายเดือน
joined_date
2025-04    138
2025-05    140
2025-06    142
2025-07    142
2025-08    144
2025-09    148
2025-10    145
2025-11    175
2025-12    197
2026-01    190
2026-02    161
2026-03    189
2026-04    201
2026-05    200
2026-06    182
2026-07    210
2026-08    215
2026-09     81
Freq: M


### 3.4 `phone` · รูปแบบ และใช้แยกคนได้ไหม

In [8]:
print("รูปแบบ:", df["phone"].map(pattern).value_counts().to_dict())
dup_phone = df[df["phone"].duplicated(keep=False)].sort_values("phone")
print(f"เบอร์ซ้ำ: {dup_phone['phone'].nunique():,} เบอร์ · {len(dup_phone):,} แถว")
dup_phone.head(6)

รูปแบบ: {'999-xxx-9999': 3000}
เบอร์ซ้ำ: 49 เบอร์ · 99 แถว


,customer_id,nickname,gender,age_group,home_branch_id,joined_date,phone
2206,C02207,ทิพย์,ชาย,35-44,B02,2025-04-14 00:00:00,081-xxx-4308
2247,C02248,อัญชลี,ชาย,35-44,B02,2026-03-15 00:00:00,081-xxx-4308
2178,C02179,อัญชลี,หญิง,25-34,B01,2025-06-24 00:00:00,081-xxx-5843
2842,C02843,ธีร,หญิง,25-34,B05,2026-05-20 00:00:00,081-xxx-5843
1906,C01907,ศิริ,หญิง,25-34,B03,2026-03-31 00:00:00,081-xxx-6451
883,C00884,ภัทร,ชาย,45-54,B01,2025-09-07 00:00:00,081-xxx-6451


เบอร์ถูกปิดบัง 3 หลักกลาง (`xxx`) เหลือแค่ prefix กับ 4 หลักท้าย ซึ่งมีแค่ประมาณ 80,000 แบบ
ลูกค้า 3,000 คนจึงชนกันโดยบังเอิญราว 50 คู่ตามหลักความน่าจะเป็น และเมื่อดูชื่อเล่น เพศ และอายุของคู่ที่ซ้ำก็ต่างกันเกือบทั้งหมด
**สรุป: เบอร์ซ้ำไม่ได้แปลว่าเป็นลูกค้าซ้ำ และใช้ `phone` เป็นตัวระบุตัวตนไม่ได้**

In [9]:
same_person = dup_phone.groupby("phone").agg(nick=("nickname", "nunique"), gender=("gender", "nunique"), age=("age_group", "nunique"))
n_same = int(((same_person == 1).all(axis=1)).sum())
print(f"กลุ่มเบอร์ซ้ำที่ชื่อเล่น เพศ อายุ ตรงกันหมด (อาจเป็นคนเดียวกัน): {n_same} จาก {len(same_person)} กลุ่ม")
note("phone", "เบอร์ซ้ำ (ถูกปิดบัง xxx)", len(dup_phone), "ต่ำ",
     f"{dup_phone['phone'].nunique()} เบอร์ ส่วนใหญ่เป็นคนละคน ห้ามใช้ phone ระบุตัวตนหรือลบแถวซ้ำ")

กลุ่มเบอร์ซ้ำที่ชื่อเล่น เพศ อายุ ตรงกันหมด (อาจเป็นคนเดียวกัน): 0 จาก 49 กลุ่ม


### 3.5 `nickname` × `gender` · ข้อมูลสอดคล้องกันไหม
ชื่อเล่นไม่ได้บอกเพศเสมอ แต่ถ้าชื่ออย่าง "สมชาย" มีผู้หญิงครึ่งหนึ่ง แปลว่าคอลัมน์ใดคอลัมน์หนึ่งอาจถูกสุ่มหรือกรอกผิด

In [10]:
ct = pd.crosstab(df["nickname"], df["gender"])
ct["% หญิง"] = (ct.get("หญิง", 0) / ct.sum(axis=1) * 100).round(0)
print(f"ชื่อเล่นไม่ซ้ำ {df['nickname'].nunique()} ชื่อ จากลูกค้า {len(df):,} คน")
ct.sort_values("% หญิง")

ชื่อเล่นไม่ซ้ำ 24 ชื่อ จากลูกค้า 3,000 คน


gender,ชาย,หญิง,ไม่ระบุ,% หญิง
nickname,,,,
ชนา,65,64,8,47.0
นภัส,64,60,4,47.0
วรรณ,48,51,2,50.0
สมชาย,50,59,7,51.0
วีระ,48,53,3,51.0
ธนา,59,68,4,52.0
มานพ,48,64,10,52.0
ศิริ,52,64,5,53.0
อัญชลี,62,79,7,53.0


In [11]:
male_names = ["สมชาย", "วีระ", "มานพ", "กิตติ", "ธีร"]  # ชื่อที่คนไทยส่วนใหญ่ใช้กับผู้ชาย
n_mismatch = int(df[df["nickname"].isin(male_names)]["gender"].eq("หญิง").sum())
print(f"ชื่อผู้ชายแต่เพศเป็นหญิง: {n_mismatch:,} แถว")
note("nickname / gender", "เพศไม่สอดคล้องกับชื่อเล่น", n_mismatch, "กลาง",
     "ทุกชื่อมีหญิง 47–65% เท่า ๆ กัน คล้ายสุ่ม ระวังถ้าจะวิเคราะห์ตามเพศ")

ชื่อผู้ชายแต่เพศเป็นหญิง: 329 แถว


## ขั้นที่ 4 · ตรวจข้ามไฟล์
ข้อมูลที่ดูถูกต้องในไฟล์เดียว อาจขัดกับไฟล์อื่นได้ ถ้าไม่มีไฟล์ ให้กด Cancel แล้วข้ามได้

### 4.1 กับ `branches.xlsx` · สาขามีจริงไหม และสมัครก่อนสาขาเปิดหรือเปล่า

In [12]:
branches = load_file("branches.xlsx", required=False)
if branches is not None:
    m = df.merge(branches[["branch_id", "branch", "opened_date"]], left_on="home_branch_id", right_on="branch_id", how="left")
    n_unknown_branch = int(m["branch"].isna().sum())
    early = m[pd.to_datetime(m["joined_date"]) < pd.to_datetime(m["opened_date"])]
    print(f"home_branch_id ที่ไม่มีใน branches: {n_unknown_branch:,} · สมัครก่อนสาขาเปิด: {len(early):,}")
    print(m.groupby(["home_branch_id", "branch", "opened_date"]).agg(
        ลูกค้า=("customer_id", "size"), สมัครแรกสุด=("joined_date", lambda s: s.min()[:10])).to_string())
    note("home_branch_id", "ไม่มีใน branches.xlsx", n_unknown_branch, "—" if n_unknown_branch == 0 else "สูง")
    note("joined_date", "สมัครก่อนสาขาประจำเปิด", len(early), "—" if early.empty else "สูง")

home_branch_id ที่ไม่มีใน branches: 0 · สมัครก่อนสาขาเปิด: 0
                                                ลูกค้า สมัครแรกสุด
home_branch_id branch      opened_date                            
B01            สยาม        2023-06-01 00:00:00     776  2025-04-01
B02            สีลม        2023-09-15 00:00:00     643  2025-04-01
B03            อารีย์      2025-11-01 00:00:00     405  2025-11-01
B04            บางนา       2024-03-01 00:00:00     483  2025-04-03
B05            มหาวิทยาลัย 2024-06-01 00:00:00     693  2025-04-01


### 4.2 กับข้อมูลยอดขาย · ใช้ `sales_clean.csv` (หรือ `sales.csv` ของ Lab 1)

In [13]:
sales = load_file("sales_clean.csv", required=False)
if sales is None:
    sales = load_file("sales.csv", required=False)
if sales is not None:
    members = sales[sales["customer_id"].str.strip() != ""]
    ids = set(df["customer_id"])
    orphan = members[~members["customer_id"].isin(ids)]
    never = df[~df["customer_id"].isin(members["customer_id"])]
    first_buy = members.groupby("customer_id")["datetime"].min().str[:10]
    joined = df.set_index("customer_id")["joined_date"].str[:10]
    before_join = first_buy[first_buy < joined.reindex(first_buy.index)]
    home = df.set_index("customer_id")["home_branch_id"]
    if branches is not None:
        home = home.map(branches.set_index("branch_id")["branch"])
        at_home = (members["branch"] == members["customer_id"].map(home)).mean()
        print(f"สัดส่วนการซื้อที่สาขาประจำ: {at_home:.1%}")
    print(f"customer_id ในยอดขายที่ไม่มีในไฟล์ลูกค้า: {orphan['customer_id'].nunique():,} คน ({len(orphan):,} แถว)")
    print(f"สมาชิกที่ไม่เคยซื้อเลย: {len(never):,} คน ({len(never) / len(df):.1%})")
    print(f"ซื้อครั้งแรกก่อนวันสมัคร: {len(before_join):,} คน")
    print("\nสมาชิกที่ไม่เคยซื้อ แยกตามเดือนที่สมัคร")
    print(never["joined_date"].str[:7].value_counts().sort_index().to_string())
    note("customer_id", "อยู่ในยอดขายแต่ไม่มีในไฟล์ลูกค้า", len(orphan), "—" if orphan.empty else "สูง")
    note("customer_id", "สมาชิกที่ไม่เคยซื้อ", len(never), "ต่ำ", "ส่วนใหญ่เพิ่งสมัครช่วงท้ายข้อมูล ไม่ใช่ข้อมูลผิด")
    note("joined_date", "ซื้อครั้งแรกก่อนวันสมัคร", len(before_join), "—" if before_join.empty else "กลาง")

สัดส่วนการซื้อที่สาขาประจำ: 88.3%
customer_id ในยอดขายที่ไม่มีในไฟล์ลูกค้า: 0 คน (0 แถว)
สมาชิกที่ไม่เคยซื้อเลย: 493 คน (16.4%)
ซื้อครั้งแรกก่อนวันสมัคร: 0 คน

สมาชิกที่ไม่เคยซื้อ แยกตามเดือนที่สมัคร
joined_date
2025-04      5
2025-05      9
2025-06      9
2025-07     10
2025-08      7
2025-09      6
2025-10     10
2025-11     13
2025-12     13
2026-01     20
2026-02     16
2026-03     23
2026-04     31
2026-05     30
2026-06     41
2026-07     72
2026-08    118
2026-09     60


## ขั้นที่ 5 · สรุปปัญหา
ตารางนี้สร้างจากผลของทุกขั้นข้างบน (ไม่มีตัวเลขพิมพ์ตายตัว) แถวที่ความรุนแรงเป็น "—" คือตรวจแล้ว **ไม่พบปัญหา**

In [14]:
order = {"สูง": 0, "กลาง": 1, "ต่ำ": 2, "—": 3}
summary = pd.DataFrame(issues).sort_values("ความรุนแรง", key=lambda s: s.map(order), kind="stable").reset_index(drop=True)
summary.to_csv("customers_profiling.csv", index=False, encoding="utf-8-sig")
summary

,คอลัมน์,ปัญหา,จำนวนแถว,ความรุนแรง,หมายเหตุ
0,nickname / gender,เพศไม่สอดคล้องกับชื่อเล่น,329,กลาง,ทุกชื่อมีหญิง 47–65% เท่า ๆ กัน คล้ายสุ่ม ระวั...
1,age_group,ลูกค้าอายุต่ำกว่า 18 ปี,113,ต่ำ,ไม่ใช่ข้อมูลผิด แต่เป็นผู้เยาว์ ต้องดูเรื่องคว...
2,age_group,ค่าเป็นข้อความ เรียงตามตัวอักษรไม่ถูก,6,ต่ำ,'ต่ำกว่า 18' จะไปอยู่ท้ายสุด ต้องกำหนดลำดับเอง...
3,gender,ไม่ระบุเพศ,116,ต่ำ,ค่าที่ตั้งใจให้เลือกได้ ไม่ใช่ข้อมูลหาย
4,joined_date,เก็บเป็น datetime มีเวลา 00:00:00 ติดมา,3000,ต่ำ,เวลาไม่มีความหมาย ควรเก็บเป็นวันที่อย่างเดียว
5,phone,เบอร์ซ้ำ (ถูกปิดบัง xxx),99,ต่ำ,49 เบอร์ ส่วนใหญ่เป็นคนละคน ห้ามใช้ phone ระบุ...
6,customer_id,สมาชิกที่ไม่เคยซื้อ,493,ต่ำ,ส่วนใหญ่เพิ่งสมัครช่วงท้ายข้อมูล ไม่ใช่ข้อมูลผิด
7,(ทุกคอลัมน์),ค่าว่าง,0,—,ไม่มีค่าว่างเลย
8,(ทุกคอลัมน์),แถวซ้ำทุกคอลัมน์,0,—,
9,customer_id,รูปแบบผิดจาก C99999,0,—,


In [15]:
try:
    from google.colab import files
    files.download("customers_profiling.csv")
except ImportError:
    print("บันทึก customers_profiling.csv แล้ว")

บันทึก customers_profiling.csv แล้ว


## ขั้นที่ 6 · ตัดสินใจก่อนลงมือ

คนที่รู้ธุรกิจต้องเป็นคนตัดสิน ช่อง "ตัดสินใจ" ใส่ **ค่าเริ่มต้น** ไว้ให้แล้ว ถ้าไม่เห็นด้วยให้แก้ทั้งตารางนี้และตัวแปร `DECISIONS` ในขั้นที่ 7

| ปัญหา | ทางเลือก | ตัดสินใจ (ค่าเริ่มต้น) | เหตุผล |
|---|---|---|---|
| `joined_date` มีเวลา 00:00:00 | ตัดเหลือวันที่ / เก็บไว้ | **ตัดเหลือ `YYYY-MM-DD`** | เวลาไม่มีความหมาย และจะได้รูปแบบเดียวกับคอลัมน์ `date` ของยอดขาย |
| เบอร์โทรซ้ำ (ถูกปิดบัง) | เก็บทุกแถว / ลบแถวซ้ำ / ตัดคอลัมน์ phone ทิ้ง | **เก็บทุกแถว และตัดคอลัมน์ `phone` ออก** | เป็นคนละคน ลบแถวไม่ได้ ส่วนเบอร์เป็นข้อมูลส่วนบุคคลที่ Dashboard ไม่ได้ใช้ |
| เพศไม่สอดคล้องกับชื่อเล่น | เชื่อ gender / ไม่ใช้ gender วิเคราะห์ / ถามเจ้าของข้อมูล | **เก็บค่าเดิม ไม่แก้** | เดาเพศจากชื่อเล่นไม่ได้ ให้ระวังตอนวิเคราะห์ และถามเจ้าของข้อมูลต่อ |
| ลูกค้าอายุต่ำกว่า 18 | เก็บ / แยกกลุ่ม / ตรวจความยินยอม | **เก็บ และเพิ่มคอลัมน์ `is_minor`** | ไม่ใช่ข้อมูลผิด แต่ต้องกรองออกได้ง่ายเวลาทำการตลาด |
| สมาชิกที่ไม่เคยซื้อ | เก็บ / ตัดออก | **เก็บ** | เป็นลูกค้าจริงที่เพิ่งสมัคร การตัดสินใจนี้ทำตอนวิเคราะห์ ไม่ใช่ตอนทำความสะอาด |
| `age_group` เรียงผิด | กำหนดลำดับเอง | **เพิ่มคอลัมน์ `age_group_order` (1–6)** | CSV เก็บลำดับของ `pd.Categorical` ไม่ได้ จึงใช้ตัวเลขช่วยเรียง |
| ไม่มีชื่อสาขา มีแต่ `B01` | เพิ่มชื่อ / ไม่เพิ่ม | **เพิ่มคอลัมน์ `home_branch`** จาก `branches.xlsx` | ยอดขายใช้ชื่อสาขา ไม่ใช่รหัส จะได้ join กันได้ตรง ๆ |

**คำถามชวนคิด**
- ถ้าจะหา "ลูกค้าที่ซื้อซ้ำบ่อยที่สุดแยกตามเพศ" ผลจะเชื่อได้แค่ไหน จากสิ่งที่เห็นในขั้นที่ 3.5
- ทำไมการลบ "เบอร์ซ้ำ" ถึงอันตรายกว่าการลบ "แถวซ้ำทุกคอลัมน์"
- คอลัมน์ไหนในไฟล์นี้เป็นข้อมูลส่วนบุคคลที่ไม่ควรขึ้น Dashboard

## ขั้นที่ 7 · ทำความสะอาด

ทำงานบน `clean = df.copy()` ไม่แก้ `df` และแต่ละขั้นจะบันทึกลง `cleaning_log` ว่ากระทบกี่แถว และตัดสินใจอย่างไร

In [16]:
DECISIONS = {
    "drop_phone": True,        # ตัดคอลัมน์ phone ออก (ข้อมูลส่วนบุคคลที่ Dashboard ไม่ใช้)
    "add_is_minor": True,      # เพิ่มคอลัมน์ is_minor
    "add_branch_name": True,   # เพิ่มชื่อสาขา home_branch จาก branches.xlsx
}
AGE_ORDER = ["ต่ำกว่า 18", "18-24", "25-34", "35-44", "45-54", "55+"]

clean = df.copy()
log = []
def step(name, n, decision):
    log.append({"ขั้นตอน": name, "จำนวนแถวที่กระทบ": int(n), "การตัดสินใจ": decision})

# 1) ตัดช่องว่างหัว/ท้ายทุกคอลัมน์ (กันไว้ก่อน ถึงตอน profiling จะไม่พบ)
stripped = clean.apply(lambda s: s.str.strip())
step("ตัดช่องว่างหัว/ท้าย", (stripped != clean).any(axis=1).sum(), "strip ทุกคอลัมน์")
clean = stripped

# 2) แถวซ้ำทุกคอลัมน์ และ customer_id ซ้ำ (กันไว้ก่อน)
dup = clean.duplicated()
step("ลบแถวซ้ำทุกคอลัมน์", dup.sum(), "เก็บแถวแรก")
clean = clean[~dup]
dup_id = clean["customer_id"].duplicated(keep=False)
if dup_id.any():
    raise ValueError(f"customer_id ซ้ำแต่ข้อมูลต่างกัน {dup_id.sum()} แถว ต้องให้คนตัดสินว่าแถวไหนถูก")
step("ตรวจ customer_id ซ้ำ", 0, "ไม่พบ")

# 3) joined_date → YYYY-MM-DD
jd = pd.to_datetime(clean["joined_date"], errors="raise")
changed = clean["joined_date"] != jd.dt.strftime("%Y-%m-%d")
clean["joined_date"] = jd.dt.strftime("%Y-%m-%d")
step("joined_date เหลือแค่วันที่", changed.sum(), "ตัด 00:00:00 ออก")

# 4) age_group → ลำดับตัวเลข
unknown_age = set(clean["age_group"]) - set(AGE_ORDER)
if unknown_age:
    raise ValueError(f"พบ age_group ที่ไม่รู้จัก: {unknown_age}")
clean["age_group_order"] = clean["age_group"].map({a: i + 1 for i, a in enumerate(AGE_ORDER)})
step("เพิ่ม age_group_order", len(clean), "1 = ต่ำกว่า 18 … 6 = 55+")

# 5) ผู้เยาว์
if DECISIONS["add_is_minor"]:
    clean["is_minor"] = clean["age_group"].eq("ต่ำกว่า 18")
    step("เพิ่ม is_minor", clean["is_minor"].sum(), "เก็บแถวไว้ แค่ติดป้าย")

# 6) gender ไม่แก้
step("gender ไม่สอดคล้องกับชื่อเล่น", 0, "เก็บค่าเดิม ระวังตอนวิเคราะห์ตามเพศ")

# 7) phone
if DECISIONS["drop_phone"]:
    clean = clean.drop(columns="phone")
    step("ตัดคอลัมน์ phone", len(clean), "ข้อมูลส่วนบุคคล และใช้ระบุตัวตนไม่ได้")
else:
    step("เก็บคอลัมน์ phone", 0, "ห้ามใช้เป็นตัวระบุตัวตน")

# 8) ชื่อสาขา
if DECISIONS["add_branch_name"] and branches is not None:
    names = branches.set_index("branch_id")["branch"]
    clean["home_branch"] = clean["home_branch_id"].map(names)
    step("เพิ่ม home_branch", clean["home_branch"].notna().sum(), "map จาก branches.xlsx")
elif DECISIONS["add_branch_name"]:
    step("เพิ่ม home_branch", 0, "ข้าม: ไม่มี branches.xlsx")

clean = clean.reset_index(drop=True)
cleaning_log = pd.DataFrame(log)
print(f"{len(df):,} แถว → {len(clean):,} แถว · คอลัมน์: {', '.join(clean.columns)}")
cleaning_log

3,000 แถว → 3,000 แถว · คอลัมน์: customer_id, nickname, gender, age_group, home_branch_id, joined_date, age_group_order, is_minor, home_branch


,ขั้นตอน,จำนวนแถวที่กระทบ,การตัดสินใจ
0,ตัดช่องว่างหัว/ท้าย,0,strip ทุกคอลัมน์
1,ลบแถวซ้ำทุกคอลัมน์,0,เก็บแถวแรก
2,ตรวจ customer_id ซ้ำ,0,ไม่พบ
3,joined_date เหลือแค่วันที่,3000,ตัด 00:00:00 ออก
4,เพิ่ม age_group_order,3000,1 = ต่ำกว่า 18 … 6 = 55+
5,เพิ่ม is_minor,113,เก็บแถวไว้ แค่ติดป้าย
6,gender ไม่สอดคล้องกับชื่อเล่น,0,เก็บค่าเดิม ระวังตอนวิเคราะห์ตามเพศ
7,ตัดคอลัมน์ phone,3000,ข้อมูลส่วนบุคคล และใช้ระบุตัวตนไม่ได้
8,เพิ่ม home_branch,3000,map จาก branches.xlsx


## ขั้นที่ 8 · Verify
ต้องได้ ✅ ทุกข้อก่อน export ถ้ามี ❌ ให้คัดลอกผลทั้งหมดไปให้ AI แล้วพิมพ์ว่า "ช่วยหาสาเหตุ และอธิบายก่อนแก้"

In [17]:
def validate_customers(clean, raw):
    ok = True
    def rep(passed, msg):
        nonlocal ok
        ok &= bool(passed)
        print(("✅ " if passed else "❌ ") + msg)

    rep(len(clean) == len(raw.drop_duplicates()), f"จำนวนแถว {len(clean):,} = ข้อมูลดิบที่ไม่ซ้ำ {len(raw.drop_duplicates()):,} (ไม่มีแถวหายเงียบ ๆ)")
    rep(clean["customer_id"].is_unique, "customer_id ไม่ซ้ำ")
    rep(clean["customer_id"].str.fullmatch(r"C\d{5}").all(), "customer_id รูปแบบ C99999 ทุกแถว")
    rep(clean["joined_date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}").all(), "joined_date เป็น YYYY-MM-DD ทุกแถว")
    rep(pd.to_datetime(clean["joined_date"]).dt.year.between(2023, 2026).all(), "joined_date เป็นปี ค.ศ. ในช่วงที่ร้านเปิด")
    rep(clean["age_group_order"].notna().all(), "age_group_order ครบทุกแถว")
    rep(set(clean["gender"]) <= {"หญิง", "ชาย", "ไม่ระบุ"}, "gender มีแค่ หญิง / ชาย / ไม่ระบุ")
    rep((clean.drop(columns=["is_minor", "age_group_order"], errors="ignore") != "").all().all(), "ไม่มีค่าว่าง")
    if DECISIONS["drop_phone"]:
        rep("phone" not in clean.columns, "ไม่มีคอลัมน์ phone")
    if "home_branch" in clean.columns:
        rep(clean["home_branch"].notna().all(), "home_branch map ได้ครบทุกแถว")
    if sales is not None:
        members = set(sales.loc[sales["customer_id"].str.strip() != "", "customer_id"])
        rep(members <= set(clean["customer_id"]), "ทุก customer_id ในยอดขายยังอยู่ในไฟล์ลูกค้า")
    print("\nพร้อม export" if ok else "\nยังไม่ผ่าน อย่าเพิ่ง export")
    return ok

validate_customers(clean, df)
clean.head()

✅ จำนวนแถว 3,000 = ข้อมูลดิบที่ไม่ซ้ำ 3,000 (ไม่มีแถวหายเงียบ ๆ)
✅ customer_id ไม่ซ้ำ
✅ customer_id รูปแบบ C99999 ทุกแถว
✅ joined_date เป็น YYYY-MM-DD ทุกแถว
✅ joined_date เป็นปี ค.ศ. ในช่วงที่ร้านเปิด
✅ age_group_order ครบทุกแถว
✅ gender มีแค่ หญิง / ชาย / ไม่ระบุ
✅ ไม่มีค่าว่าง
✅ ไม่มีคอลัมน์ phone
✅ home_branch map ได้ครบทุกแถว
✅ ทุก customer_id ในยอดขายยังอยู่ในไฟล์ลูกค้า

พร้อม export


,customer_id,nickname,gender,age_group,home_branch_id,joined_date,age_group_order,is_minor,home_branch
0,C00001,กมล,หญิง,18-24,B02,2026-03-03,2,False,สีลม
1,C00002,กิตติ,ชาย,25-34,B04,2026-01-18,3,False,บางนา
2,C00003,ปิยะ,หญิง,18-24,B04,2025-05-04,2,False,บางนา
3,C00004,ปิยะ,หญิง,18-24,B05,2026-05-05,2,False,มหาวิทยาลัย
4,C00005,รัตนา,หญิง,25-34,B04,2026-07-10,3,False,บางนา


## ขั้นที่ 9 · Export
`customers_clean.csv` คือไฟล์ที่ใช้ต่อ ส่วน `customers_cleaning_log.csv` คือหลักฐานว่าทำอะไรกับข้อมูลไปบ้าง ให้แนบไว้ใน repo ด้วย

In [18]:
clean.to_csv("customers_clean.csv", index=False, encoding="utf-8-sig")
cleaning_log.to_csv("customers_cleaning_log.csv", index=False, encoding="utf-8-sig")
print("บันทึก customers_clean.csv และ customers_cleaning_log.csv แล้ว")

try:
    from google.colab import files
    files.download("customers_clean.csv")
    files.download("customers_cleaning_log.csv")
except ImportError:
    pass

บันทึก customers_clean.csv และ customers_cleaning_log.csv แล้ว


### นำไปใช้ต่อ
- วาง `customers_clean.csv` ไว้ใน `public/` ของ Dashboard แล้ว join กับยอดขายด้วย `customer_id`
- ใช้ `age_group_order` เรียงกลุ่มอายุบนกราฟ และกรอง `is_minor` ออกก่อนทำแคมเปญการตลาด
- ห้ามสรุปเรื่องเพศจากข้อมูลนี้จนกว่าเจ้าของข้อมูลจะยืนยันว่าคอลัมน์ `gender` ถูกต้อง